In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
OUTPUT = TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
Y = CONFIG['last_year']

In [3]:
ranks = pd.read_csv(OUTPUT/'rank_correlations.csv')
distributions = pd.read_csv(OUTPUT/'distribution_summary.csv')
distances = pd.read_csv(OUTPUT/'distribution_distances.csv')
moments = pd.read_csv(OUTPUT/'annual_moments.csv')
qq_summary = pd.read_csv(OUTPUT/'window_qq_summary.csv')

In [4]:
# common late window; lag-3 and lag-5 pairs fully generated for AR(6,6)
annual = ranks.loc[ranks.measure.eq('annual')]
empirical = annual.loc[annual.source.eq('empirical_complete')]
rows = []
for source in ['empirical_complete'] + CONFIG['models']:
    selected = annual.loc[annual.source.eq(source)]
    row = {'source':source, 'n':int(selected.n.iloc[0]),
           'generated_from_age':6 if source=='ar6_6' else 1 if source!='empirical_complete' else np.nan,
           'fit_cohort':'complete careers' if source=='se_hurdle_s' else 'available observations' if source!='empirical_complete' else 'complete careers'}
    for reference in ['age_0','lag_3','lag_5']:
        row[f'{reference}_rho20'] = selected.loc[selected.reference.eq(reference)&selected.age.eq(Y),'rho'].iloc[0]
        late = selected.loc[selected.reference.eq(reference)&selected.age.between(11,Y)]
        matched = late.merge(empirical.loc[empirical.reference.eq(reference),['age','rho']],on='age',suffixes=('_sim','_emp'))
        row[f'{reference}_rmse11_20'] = np.sqrt(np.mean((matched.rho_sim-matched.rho_emp)**2))
    totals = distributions.loc[distributions.source.eq(source)&distributions.window.eq('career_total')].iloc[0]
    last_four = distributions.loc[distributions.source.eq(source)&distributions.window.eq('last_four')].iloc[0]
    last_shape = qq_summary.loc[qq_summary.source.eq(source)&qq_summary.window.eq('last_four')].iloc[0]
    career_shape = qq_summary.loc[qq_summary.source.eq(source)&qq_summary.window.eq('career_total')].iloc[0]
    generated = distributions.loc[distributions.source.eq(source)&distributions.window.eq('generated_common')].iloc[0]
    final = moments.loc[moments.source.eq(source)&moments.age.eq(Y)].iloc[0]
    row.update({'lognormality_window':'last_four', 'last_four_start_age':Y-3, 'last_four_end_age':Y,
                'mean_last_four_total':last_four.mean_total, 'gini_last_four_total':last_four.gini_total,
                'last_four_n_positive':int(last_shape.n_positive),
                'last_four_zero_fraction':1-last_shape.n_positive/last_shape.n_total,
                'last_four_qq_r':last_shape.qq_r, 'last_four_qq_r2':last_shape.qq_r2,
                'last_four_ks_D':last_shape.ks_D, 'career_total_qq_r2':career_shape.qq_r2,
                'mean_career_total':totals.mean_total, 'gini_career_total':totals.gini_total,
                'mean_total6_20':generated.mean_total, 'gini_total6_20':generated.gini_total,
                'final_mean':final['mean'], 'final_zero_fraction':final.zero_fraction})
    if source!='empirical_complete':
        distance = distances.loc[distances.source.eq(source)&distances.diagnostic.eq('generated_common')].iloc[0]
        row['total6_20_ks_D'] = distance.ks_D
    rows.append(row)
comparison = pd.DataFrame(rows)
comparison.to_csv(OUTPUT/'model_comparison.csv',index=False)
display(comparison)

,source,n,generated_from_age,fit_cohort,age_0_rho20,age_0_rmse11_20,lag_3_rho20,lag_3_rmse11_20,lag_5_rho20,lag_5_rmse11_20,...,last_four_qq_r2,last_four_ks_D,career_total_qq_r2,mean_career_total,gini_career_total,mean_total6_20,gini_total6_20,final_mean,final_zero_fraction,total6_20_ks_D
0,empirical_complete,510,NaN,complete careers,0.187294,0.000000,0.653156,0.000000,0.582696,0.000000,...,0.975473,0.060820,0.991906,152.995259,0.360430,111.126608,0.388352,6.789832,0.141176,NaN
1,arw4,10000,1.0,available observations,0.001876,0.226145,0.202827,0.414606,0.077234,0.453604,...,0.991270,0.026109,0.999731,147.036215,0.183385,104.726944,0.204965,6.666496,0.000000,0.224808
2,lognormal_grw,10000,1.0,available observations,0.139936,0.055432,0.833387,0.200707,0.741913,0.180233,...,0.983866,0.045595,0.957396,56998.145495,0.984804,56757.164158,0.986042,10898.075199,0.109500,0.392784
3,ar1_s,10000,1.0,available observations,0.001461,0.236766,0.245662,0.358143,0.100119,0.419343,...,0.991569,0.023438,0.999467,155.612271,0.287743,110.842995,0.345925,6.983392,0.025000,0.068778
4,hurdle_ar1_s_p,10000,1.0,available observations,-0.011916,0.220046,0.308526,0.317960,0.136052,0.391821,...,0.932259,0.079963,0.987290,135.411339,0.255373,96.931943,0.296480,6.281482,0.156600,0.106145
5,ar6_6,10000,6.0,available observations,0.129372,0.060080,0.592000,0.049308,0.530811,0.034095,...,0.935821,0.102183,0.934332,136.980226,0.329544,95.644503,0.363410,5.737718,0.164900,0.075892
6,se_hurdle_s,10000,1.0,complete careers,0.159485,0.038154,0.610686,0.043352,0.553039,0.026413,...,0.973258,0.058832,0.967322,151.488333,0.341177,109.811206,0.377233,7.170356,0.143400,0.046437


In [5]:
# final-four-year sums are primary; whole-career sums retained as a supplement
lognormality = qq_summary.loc[qq_summary.source.isin(['empirical_complete']+CONFIG['models'])
                             & qq_summary.window.isin(['last_four','career_total'])].copy()
lognormality['role'] = lognormality.window.map({'last_four':'primary','career_total':'supplement'})
lognormality['start_age'] = np.where(lognormality.window.eq('last_four'),Y-3,0)
lognormality['end_age'] = Y
lognormality['n_zero'] = lognormality.n_total-lognormality.n_positive
lognormality['zero_fraction'] = lognormality.n_zero/lognormality.n_total
lognormality = lognormality.merge(distributions[['source','window','mean_total','gini_total']],
                                  on=['source','window'])
columns = ['source','window','role','start_age','end_age','n_total','n_positive','n_zero','zero_fraction',
           'mean_total','gini_total','loc','fit_scale','qq_r','qq_r2','ks_D','prefix_included']
lognormality = lognormality[columns].sort_values(['start_age','source'], ascending=[False,True])
lognormality.to_csv(OUTPUT/'lognormality_summary.csv',index=False)
display(lognormality.loc[lognormality.role.eq('primary')])

,source,window,role,start_age,end_age,n_total,n_positive,n_zero,zero_fraction,mean_total,gini_total,loc,fit_scale,qq_r,qq_r2,ks_D,prefix_included
6,ar1_s,last_four,primary,17,20,10000,9999,1,0.000100,27.947797,0.507315,4.089933,1.507395,0.995776,0.991569,0.023438,False
10,ar6_6,last_four,primary,17,20,10000,9648,352,0.035200,23.469428,0.442298,4.028420,1.499162,0.967378,0.935821,0.102183,False
2,arw4,last_four,primary,17,20,10000,10000,0,0.000000,26.897091,0.318369,4.494730,0.890796,0.995625,0.991270,0.026109,False
0,empirical_complete,last_four,primary,17,20,510,500,10,0.019608,28.127164,0.467120,4.240414,1.441332,0.987660,0.975473,0.060820,False
8,hurdle_ar1_s_p,last_four,primary,17,20,10000,9855,145,0.014500,25.129366,0.447935,4.072660,1.529635,0.965536,0.932259,0.079963,False
4,lognormal_grw,last_four,primary,17,20,10000,9825,175,0.017500,41454.510583,0.991900,5.608451,4.650150,0.991900,0.983866,0.045595,False
12,se_hurdle_s,last_four,primary,17,20,10000,9741,259,0.025900,28.800348,0.460437,4.306557,1.418203,0.986538,0.973258,0.058832,False
